# Self-Reflection & Critique: Engineering Autonomous LLM Reflection
### Practical implementations of every concept from the BIA deck

**How LLMs diagnose, refine, and improve outputs before agents scale up.**

This notebook turns each slide of the PDF into runnable code:

| # | Section | Slide concept |
|---|---------|---------------|
| 1 | Setup & helper functions | — |
| 2 | The motivating problem | *The Brief vs The First Draft* |
| 3 | Self-Reflection vs Critique | *The useful unit is "what should change, and why?"* |
| 4 | The 5 ingredients of a useful critique | *Quality Check pentagon* |
| 5 | Weak vs Strong critique prompts | *Critique quality depends on review criteria* |
| 6 | **Pattern 1: Self-Refine** | Generator → Feedback → Refiner loop |
| 7 | **Pattern 2: Reflexion** | Language feedback as memory, not gradients |
| 8 | Reflection Memory Triage | Store vs Avoid + Refresh protocol |
| 9 | **Pattern 3: Evaluator-Generator** | Threshold gate + structured scores |
| 10 | The Evaluator Engine | Rubric → JSON (`score`, `passed`, `issues`, `revision_instructions`) |
| 11 | **Pattern 4: Principles-Based Critique** | The Constitution + Critic Agent |
| 12 | Stopping Rules (Circuit Breakers) | Pass threshold, max iterations, no-improvement, escalation |
| 13 | **Capstone: The Writing Critic Agent** | Rubric + Constitution + Refiner + Trace |
| 14 | The Blueprint Matrix + exercises | Choosing your engine |

> **Core Directive (from the deck):** *Make quality improvement systematic, not accidental.*


### 1. Setup & helper functions 

In [2]:
import os, json, re, textwrap
from dataclasses import dataclass, field
from typing import Optional
from litellm import completion
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
MODEL = "gpt-4o-mini"  

In [5]:
def llm(prompt: str, system: str = "You are a helpful assistant.",
        temperature: float = 0.7, json_mode: bool = False) -> str:
    """Single LLM call. json_mode=True nudges + parses strict JSON output."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    resp = completion(model=MODEL, messages=messages, temperature=temperature)
    return resp.choices[0].message.content.strip()


In [11]:
def llm_json(prompt: str, system: str, temperature: float = 0.0) -> dict:
    """LLM call that must return JSON. Strips markdown fences and parses."""
    raw = llm(prompt, system=system + "\nRespond ONLY with valid JSON. No preamble, no markdown fences.",
              temperature=temperature)
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # one repair attempt — a common production trick
        repaired = llm(f"Fix this into strict valid JSON, output only the JSON:\n{raw}",
                       system="You repair malformed JSON.", temperature=0.0)
        repaired = re.sub(r"^```(?:json)?|```$", "", repaired.strip(), flags=re.MULTILINE).strip()
        return json.loads(repaired)


def show(text, width=100):
    print(textwrap.fill(text, width=width, replace_whitespace=False))

In [ ]:
show(llm_json("Hi", "You are helpful assitance to answer any query"))

{'response': 'Hello! How can I assist you today?'}

In [13]:
show(llm("Hi", "You are helpful assitance to answer any query"))

Hello! How can I assist you today?


### 2 The motivating problem


*"Is there an answer?" vs "Does this answer satisfy the brief?"*

**The Brief:** *Write a short email inviting working professionals to a weekend GenAI workshop. Tone: credible, specific, not hype-heavy.*

A single-shot LLM call reliably produces **an** answer. The deck's annotated first draft shows exactly why that's not enough:


| First draft fragment | Diagnosed defect |
|---|---|
| "Join our amazing AI workshop!" | Audience is too broad |
| "Learn everything about GenAI" | No concrete outcome |
| "...become an expert" | Overpromises expertise |
| "...transform your career." | No date / format / CTA |


> **Reflection starts when we stop asking "is there an answer?" and start asking "does this answer satisfy the actual brief?"**


Let's reproduce the failure first.



In [14]:
BRIEF = "Write a short email inviting working professionals to a weekend GenAI workshop"

first_draft = llm(f"Write an email for this: {BRIEF}", temperature=1.0)
show(first_draft)


Subject: Join Us for a Weekend GenAI Workshop!

Dear [Recipient's Name],

We are excited to invite
you to our upcoming GenAI Workshop, designed specifically for working professionals looking to
enhance their skills in this rapidly evolving field. 

**Date:** [Insert Date]  
**Time:** [Insert
Time]  
**Location:** [Insert Location/Online Platform]

This workshop will cover the fundamentals
of Generative AI, including practical applications and hands-on sessions that will empower you to
leverage AI in your work effectively. 

Whether you're new to the topic or seeking to deepen your
understanding, this workshop is the perfect opportunity to learn, network, and innovate.

Please
RSVP by [Insert RSVP Deadline] to secure your spot.

Looking forward to seeing you there!

Best
regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]  
[Your Organization]


Run the cell a few times. You'll typically see the exact defect classes the deck highlights — hype adjectives, vague outcomes, missing logistics. The rest of this notebook is about **catching and fixing those defects systematically**.

### Where this fits in the reasoning arc

```
1. Prompt (clear task) → 2. Reason (structured path) → 3. Act (tool/output)
        ↑                                                      │
        └──────── 5. Revise (improve) ← 4. Critique (detect gaps)
```

Steps 4 and 5 are the subject of this notebook. The dashed arrows in the deck matter: critique can route back to the **prompt** (re-specify the task), not only to the draft.


---
## 3. Self-Reflection vs Critique — precise definitions

The deck draws a jigsaw between two interlocking concepts:

- **Self-reflection** — *an LLM-generated review of an output against the task goal, constraints, and quality criteria.* (The act of looking.)
- **Critique** — *a diagnosis that names specific defects and gives revision instructions.* (The actionable artifact.)

> **The useful unit is not "confidence." The useful unit is "what should change, and why?"**

A reflection that says *"This looks good, 8/10, I'm fairly confident"* is useless to a refiner. A critique that says *"The subject line doesn't state the date; move '25 July' into it"* is directly executable. Let's demonstrate the difference on the same draft.


In [15]:
# A confidence-style reflection (what we DON'T want)
confidence_reflection = llm(
    f"Review this email and say how confident you are that it is good:\n\n{first_draft}",
    temperature=0.3)

print("=== CONFIDENCE-STYLE REFLECTION (weak) ===")
show(confidence_reflection)

=== CONFIDENCE-STYLE REFLECTION (weak) ===
The email you provided is well-structured and effectively communicates the key details about the
GenAI workshop. Here are some strengths and areas for improvement:

### Strengths:
1. **Clear
Subject Line**: The subject line is engaging and clearly states the purpose of the email.
2.
**Professional Tone**: The tone is appropriate for a professional audience, conveying excitement
while remaining formal.
3. **Essential Details**: The email includes important information such as
the date, time, and purpose of the workshop.
4. **Call to Action**: The RSVP request provides a
clear next step for the recipient.

### Areas for Improvement:
1. **Personalization**: Ensure that
the recipient's name is inserted correctly to make the email feel more personalized.
2. **Specific
Details**: Make sure to fill in the placeholders (date, time, location, RSVP deadline) before
sending the email. Leaving these blank can lead to confusion.
3. **Additional Information

In [16]:
change_critique = llm(
    f"""Here is a brief and a draft.

BRIEF:
{BRIEF}

DRAFT:
{first_draft}

List the specific defects (what is missing, incorrect, vague, or risky relative to the brief),
and for each defect give a concrete revision instruction a writer could apply mechanically.""",
    temperature=0.3)

print("=== CHANGE-ORIENTED CRITIQUE (strong) ===")
show(change_critique)

=== CHANGE-ORIENTED CRITIQUE (strong) ===
Here are the specific defects identified in the draft, along with concrete revision instructions for
each:

1. **Missing Personalization**:
   - **Defect**: The draft uses a generic greeting ("Dear
[Recipient's Name]") without any personalization or context.
   - **Revision Instruction**: Replace
"[Recipient's Name]" with a specific name or use a more personalized greeting, such as "Dear
Colleagues" or "Dear [Team/Department Name]".

2. **Incomplete Date and Time Information**:
   -
**Defect**: The placeholders for date and time ("[Insert Date]" and "[Insert Time]") are not filled
in.
   - **Revision Instruction**: Insert the specific date and time of the workshop in place of the
placeholders.

3. **Vague Location Information**:
   - **Defect**: The location is indicated as a
placeholder ("[Insert Location/Online Platform]") without specifics.
   - **Revision Instruction**:
Provide the exact location or specify the online platform (e.g., Zoom, 

---
## 4. A Useful Critique Has Five Ingredients

The deck's pentagon, as a data structure. Every quality check should carry:

1. **Task goal** — what the output is supposed to achieve
2. **Constraints** — format, length, audience, tone, safety limits
3. **Rubric** — *observable* criteria for quality
4. **Specific defects** — what is missing, incorrect, vague, or risky
5. **Revision instructions** — concrete changes the refiner can apply

Ingredients 1–3 are the **inputs** to a quality check; 4–5 are its **outputs**. Encoding this as dataclasses makes the contract explicit and reusable across all four patterns.


In [19]:
class Product:
    # name: str
    # price: float
    # quantity: int = 0  # Default value

    def __init__(self, name, price, product):
        self.name= name
        self.price = price
        self.product = product


    def show(self):
        print(self.name)
        print(self.price)
        print(self.product)



In [21]:
obj = Product("rajesh", 1.2, "genai")

In [22]:
obj.show()

rajesh
1.2
genai


In [24]:
feedback = {
  "feedback": [
    {
      "issue": "Missing Personalization",
      "defect": "The draft uses a generic greeting (\"Dear [Recipient's Name]\") without any personalization or context.",
      "revision_instruction": "Replace \"[Recipient's Name]\" with a specific name or use a more personalized greeting, such as \"Dear Colleagues\" or \"Dear [Team/Department Name]\"."
    },
    {
      "issue": "Incomplete Date and Time Information",
      "defect": "The placeholders for date and time (\"[Insert Date]\" and \"[Insert Time]\") are not filled in.",
      "revision_instruction": "Insert the specific date and time of the workshop in place of the placeholders."
    },
    {
      "issue": "Vague Location Information",
      "defect": "The location is indicated as a placeholder (\"[Insert Location/Online Platform]\") without specifics.",
      "revision_instruction": "Provide the exact location or specify the online platform (e.g., Zoom, Microsoft Teams) where the workshop will be held."
    },
    {
      "issue": "Lack of RSVP Details",
      "defect": "The RSVP deadline is a placeholder (\"[Insert RSVP Deadline]\") and lacks specifics.",
      "revision_instruction": "Insert a specific date and time for the RSVP deadline to encourage timely responses."
    },
    {
      "issue": "Missing Call to Action",
      "defect": "While there is an RSVP request, it could be more compelling and direct.",
      "revision_instruction": "Strengthen the call to action by adding a phrase like \"Don't miss out on this opportunity—reserve your spot today!\" before the RSVP line."
    },
    {
      "issue": "No Mention of Benefits or Outcomes",
      "defect": "The benefits of attending the workshop are somewhat vague and could be more compelling.",
      "revision_instruction": "Add specific outcomes or skills participants will gain, such as \"You will learn how to implement AI tools in your projects, improve efficiency, and drive innovation in your workplace.\""
    },
    {
      "issue": "Missing Contact Information",
      "defect": "The contact information is indicated as a placeholder (\"[Your Contact Information]\") and may not include all necessary details.",
      "revision_instruction": "Ensure that your full contact information is included, such as your phone number and email address, to facilitate communication."
    },
    {
      "issue": "Subject Line Could Be More Engaging",
      "defect": "The subject line is straightforward but could be more engaging to attract attention.",
      "revision_instruction": "Revise the subject line to something like \"Unlock the Future: Join Our Weekend GenAI Workshop!\" to create excitement."
    }
  ]
}

In [28]:
feedback['feedback'][0]['defect']

'The draft uses a generic greeting ("Dear [Recipient\'s Name]") without any personalization or context.'

In [23]:
@dataclass
class TaskSpec:
    """Ingredients 1-3: what the critic needs BEFORE it can judge anything."""
    goal: str                          # 1. Task goal
    constraints: list[str]             # 2. Constraints
    rubric: dict[str, str]             # 3. Rubric: criterion -> observable definition

    def render(self) -> str:
        lines = [f"TASK GOAL: {self.goal}", "CONSTRAINTS:"]
        lines += [f"  - {c}" for c in self.constraints]
        lines.append("RUBRIC (observable criteria):")
        lines += [f"  - {k}: {v}" for k, v in self.rubric.items()]
        return "\n".join(lines)



In [29]:
@dataclass
class Critique:
    """Ingredients 4-5: what the critic must produce."""
    score: int                         # 0-10, enables thresholds
    passed: bool                       # routes the loop
    defects: list[str]                 # 4. Specific defects (traceable debugging evidence)
    revision_instructions: str         # 5. Concrete changes the refiner can apply


In [30]:
EMAIL_SPEC = TaskSpec(
    goal="Invite working professionals to a weekend GenAI workshop and get them to register.",
    constraints=[
        "Short: under 160 words",
        "Audience: mid-career tech professionals (data/engineering) in India",
        "Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'",
        "Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)",
        "Must include: exactly one concrete learning outcome",
        "Must include: a clear CTA with a link placeholder [REGISTER_LINK]",
    ],
    rubric={
        "clarity":       "A reader knows what/when/where within 5 seconds",
        "specificity":   "Names a concrete skill/outcome, not 'learn everything'",
        "factual_risk":  "No overpromises ('become an expert', guaranteed jobs)",
        "audience_fit":  "References the reader's actual work context",
        "cta":           "One unambiguous next action with the link placeholder",
    },
)

print(EMAIL_SPEC.render())



TASK GOAL: Invite working professionals to a weekend GenAI workshop and get them to register.
CONSTRAINTS:
  - Short: under 160 words
  - Audience: mid-career tech professionals (data/engineering) in India
  - Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'
  - Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)
  - Must include: exactly one concrete learning outcome
  - Must include: a clear CTA with a link placeholder [REGISTER_LINK]
RUBRIC (observable criteria):
  - clarity: A reader knows what/when/where within 5 seconds
  - specificity: Names a concrete skill/outcome, not 'learn everything'
  - factual_risk: No overpromises ('become an expert', guaranteed jobs)
  - audience_fit: References the reader's actual work context
  - cta: One unambiguous next action with the link placeholder


In [31]:
EMAIL_SPEC.constraints

['Short: under 160 words',
 'Audience: mid-career tech professionals (data/engineering) in India',
 "Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'",
 'Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)',
 'Must include: exactly one concrete learning outcome',
 'Must include: a clear CTA with a link placeholder [REGISTER_LINK]']

---
## 5. Weak vs Strong Critique Prompts

The deck's side-by-side:

**Weak:** `"Check this and improve it."`
- ❌ no criteria &nbsp; ❌ no role separation &nbsp; ❌ no required output format &nbsp; ❌ no stopping signal

**Strong:** *"Evaluate the draft against the brief. Score clarity, specificity, factual risk, audience fit, and CTA. Return JSON with score, issues, and revision instructions."*
- ✅ task goal & constraints &nbsp; ✅ rubric &nbsp; ✅ defects & revision instructions (in a machine-parseable format)

> **Critique quality depends more on the review criteria than on the word "critique."**

Run both against the same draft and compare.


In [32]:
# WEAK critique prompt
weak = llm(f"Check this and improve it:\n\n{first_draft}", temperature=0.3)
print("=== WEAK PROMPT OUTPUT ===")
show(weak[:800])

=== WEAK PROMPT OUTPUT ===
Subject: Join Us for an Exciting Weekend GenAI Workshop!

Dear [Recipient's Name],

We are thrilled
to invite you to our upcoming GenAI Workshop, tailored specifically for professionals eager to
enhance their skills in this dynamic and rapidly evolving field.

**Date:** [Insert Date]
**Time:** [Insert Time]  
**Location:** [Insert Location/Online Platform]

During this workshop, you
will explore the fundamentals of Generative AI, engage in practical applications, and participate in
hands-on sessions designed to empower you to effectively integrate AI into your work. 

Whether you
are new to Generative AI or looking to deepen your expertise, this workshop offers a fantastic
opportunity to learn, network, and innovate alongside like-minded professionals.

Please RSVP by
[Insert RSVP Deadli


In [33]:
# STRONG critique prompt — criteria + role separation + output format + stopping signal
CRITIC_SYSTEM = "You are a strict writing critic. You never rewrite the draft; you only diagnose and instruct."

def strong_critique(draft: str, spec: TaskSpec, threshold: int = 8) -> Critique:
    prompt = f"""{spec.render()}

DRAFT:
\"\"\"{draft}\"\"\"

Evaluate the draft against the task goal, constraints, and rubric above.
Return JSON with exactly these keys:
  "score": integer 0-10 (10 = fully satisfies brief),
  "passed": boolean (true only if score >= {threshold} AND all 'Must include' constraints are met),
  "defects": array of short strings, each naming ONE specific defect,
  "revision_instructions": a single string of concrete, mechanical edits."""
    data = llm_json(prompt, system=CRITIC_SYSTEM)
    return Critique(**{k: data[k] for k in ("score", "passed", "defects", "revision_instructions")})

crit = strong_critique(first_draft, EMAIL_SPEC)

print("=== STRONG PROMPT OUTPUT (parsed Critique object) ===")
print(f"score  = {crit.score}   passed = {crit.passed}")
print("defects:")
for d in crit.defects: print("  •", d)
print("\nrevision_instructions:")
show(crit.revision_instructions)


=== STRONG PROMPT OUTPUT (parsed Critique object) ===
score  = 4   passed = False
defects:
  • Missing specific date, time, and location details.
  • Lacks a concrete learning outcome.
  • No clear call to action with a link placeholder.
  • Tone is slightly hype-heavy with phrases like 'excited to invite' and 'perfect opportunity'.
  • Does not reference the audience's work context.

revision_instructions:
Insert the specific date (Sat 25 July 2026), time (10am-5pm), and location (Bengaluru). Include a
concrete learning outcome related to Generative AI. Replace the RSVP request with a clear call to
action that includes the link placeholder [REGISTER_LINK]. Adjust the tone to be more credible and
specific.


Compare the four errors of the weak prompt against what we just fixed:

| Weak-prompt error | How the strong prompt fixes it |
|---|---|
| No criteria | `spec.render()` injects goal + constraints + rubric |
| No role separation | Critic system prompt: *"You never rewrite the draft"* |
| No required output format | Strict JSON schema, parsed into a `Critique` dataclass |
| No stopping signal | `passed` boolean tied to an explicit threshold |

The weak prompt also **conflates critic and refiner** — it rewrites the draft, so you get neither a reusable diagnosis nor a controlled revision. Role separation is the seed of Pattern 3.


---
## 6. Pattern 1: Self-Refine

Three roles, one loop:

```
        ┌──────────────────────────────┐
        │  GENERATOR: creates initial  │
        │  answer                      │
        └──────────────┬───────────────┘
                       ▼
        ┌──────────────────────────────┐
        │  FEEDBACK PROVIDER: finds    │
        │  issues and suggests fixes   │
        └──────────────┬───────────────┘
                       ▼
        ┌──────────────────────────────┐
        │  REFINER: produces improved  │──▶ loop back to feedback
        │  answer                      │    until stop condition
        └──────────────────────────────┘
```

All three roles can be **the same model with different prompts** — that's the "self" in Self-Refine (Madaan et al., 2023).

**Fit guidance from the deck:**

| ✅ Good fit (checklist-style) | ⚠️ Weak fit alone |
|---|---|
| Drafting, editing, summaries, format repair, checklist-style reviews | Unknown factual truth, hidden math errors, security-sensitive outputs |

> **Rule of thumb: Reflection improves alignment to *stated criteria*; it does not create *missing evidence*.**

If the model doesn't know a fact, reflecting harder won't conjure it — it may just make the hallucination more confident. (That's why the stopping rules in Section 12 include *escalate to human review*.)


In [34]:
def self_refine(brief: str, spec: TaskSpec, max_iterations: int = 3,
                threshold: int = 8, verbose: bool = True):
    """Pattern 1: Generator -> Feedback -> Refiner loop with a stop condition."""
    trace = []

    # --- GENERATOR ---
    draft = llm(f"{spec.render()}\n\nBRIEF:\n{brief}\n\nWrite the email.",
                system="You are a marketing copywriter.", temperature=0.8)

    for i in range(1, max_iterations + 1):
        # --- FEEDBACK PROVIDER ---
        crit = strong_critique(draft, spec, threshold=threshold)
        trace.append({"iteration": i, "score": crit.score,
                      "n_defects": len(crit.defects), "defects": crit.defects})
        if verbose:
            print(f"[iter {i}] score={crit.score} passed={crit.passed} "
                  f"defects={len(crit.defects)}")

        if crit.passed:                                   # stopping signal
            break

        # --- REFINER ---
        draft = llm(
            f"""{spec.render()}

CURRENT DRAFT:
\"\"\"{draft}\"\"\"

CRITIQUE — apply these revision instructions exactly, change nothing else:
{crit.revision_instructions}

Output only the revised email.""",
            system="You are a careful editor. You apply instructions; you do not re-invent.",
            temperature=0.4)

    return draft, trace


final_email, trace = self_refine(BRIEF, EMAIL_SPEC)
print("\n=== FINAL EMAIL ===")
show(final_email)

[iter 1] score=8 passed=True defects=0

=== FINAL EMAIL ===
Subject: Join Us for a Weekend GenAI Workshop in Bengaluru

Dear [Recipient’s Name],

We invite you
to participate in our GenAI workshop tailored for mid-career tech professionals. Join us on
Saturday, July 25, 2026, from 10 AM to 5 PM in Bengaluru for an in-depth exploration of Generative
AI applications in data engineering and analytics.

During this in-person workshop, you will gain
hands-on experience in building a generative AI model, enabling you to apply these techniques
directly to your work. 

Don’t miss this opportunity to enhance your skills and network with peers
in your field. 

[REGISTER_LINK]

Best regards,  
[Your Name]  
[Your Organization]  
[Your Contact
Information]


Pattern 2: Reflexion — *language feedback, not gradient updates*

```
ORDINARY RETRY:   attempt → fail → ask again          ──▶ (repeats the same mistake)

REFLEXION LOOP:   attempt → feedback → MEMORY NODE ───▶ next attempt
                                       (reflection)        (informed by lesson)
```

In [57]:
# A task with an EXTERNAL verifier: unit tests. (Ground-truth feedback, not LLM opinion.)
CODING_TASK = """Write a Python function `parse_price(s: str) -> float` that parses Indian-format
price strings into floats. Examples it must handle:
  "₹1,23,456.50" -> 123456.50      (Indian digit grouping)
  "Rs. 999"      -> 999.0
  " 1,000 "      -> 1000.0
  "₹0"           -> 0.0
Return only the function code, no explanation."""


TESTS = [("₹1,23,456.50", 123456.50), ("Rs. 999", 999.0), (" 1,000 ", 1000.0), ("₹0", 0.0)]




def run_tests(code_str: str):
    """External verifier. Returns (all_passed, feedback_string)."""
    ns = {}
    try:
        cleaned = re.sub(r"^```(?:python)?|```$", "", code_str.strip(), flags=re.MULTILINE)
        print(cleaned)
        exec(cleaned, ns)
        fn = ns["parse_price"]
    except Exception as e:
        return False, f"Code failed to load: {type(e).__name__}: {e}"
    failures = []
    for inp, expected in TESTS:
        try:
            got = fn(inp)
            if abs(got - expected) > 1e-6:
                failures.append(f"parse_price({inp!r}) returned {got!r}, expected {expected!r}")
        except Exception as e:
            failures.append(f"parse_price({inp!r}) raised {type(e).__name__}: {e}")
    return (len(failures) == 0), ("All tests passed." if not failures else "\n".join(failures))


In [55]:
run_tests(python_func)

def parse_price(s: str) -> float:
    s = s.strip()
    s = s.replace('₹', '')
    s = s.replace('Rs.', '')
    s = s.replace('Rs', '')
    s = s.replace(',', '')
    return float(s)


(True, 'All tests passed.')

In [56]:
exec(python_func)

parse_price("Rs. 999")

999.0

In [54]:
python_func = """
def parse_price(s: str) -> float:
    s = s.strip()
    s = s.replace('₹', '')
    s = s.replace('Rs.', '')
    s = s.replace('Rs', '')
    s = s.replace(',', '')
    return float(s)
"""

exec(python_func)

print(parse_price("₹1,23,456.50"))


123456.5


In [44]:
def ordinary_retry(task: str, max_attempts: int = 3):
    """Baseline: attempt -> fail -> ask again (no memory). Often repeats the same mistake."""
    for i in range(1, max_attempts + 1):
        code_out = llm(task, system="You are a Python developer.", temperature=0.8)
        print(code_out)
        ok, fb = run_tests(code_out)
        print(f"[ordinary retry {i}] passed={ok}")
        if ok: return code_out, i
    return code_out, max_attempts

In [45]:
ordinary_retry("Write a Python function `parse_price(s: str) -> float` that parses Indian-format")

To parse a price formatted in Indian style (which typically uses commas as thousand separators and has the rupee symbol), you can create a function in Python that handles this formatting. The Indian numbering system often groups the first three digits from the right together, and then groups every two digits thereafter (for example, `12,34,567.89`).

Below is the implementation of the `parse_price` function that converts a string representation of a price in Indian format into a float:

```python
def parse_price(s: str) -> float:
    # Remove the currency symbol if it's present
    s = s.replace("₹", "").replace("INR", "").strip()
    
    # Remove any commas from the string
    s = s.replace(",", "")
    
    # Convert the string to a float
    try:
        price = float(s)
    except ValueError:
        raise ValueError(f"Invalid price format: {s}")

    return price

# Example usage:
price_str = "₹ 12,34,567.89"
parsed_price = parse_price(price_str)
print(parsed_price)  # Output: 12

('To parse prices in Indian format, we need to account for the way prices are formatted in India. Typically, prices may include commas as thousand separators and may also include a currency symbol (like ₹ or Rs).\n\nHere\'s a function that will parse a string representing a price in Indian format and return it as a float. The function will:\n\n1. Remove any currency symbols.\n2. Remove commas.\n3. Convert the result to a float.\n\nHere is the implementation:\n\n```python\ndef parse_price(s: str) -> float:\n    # Remove currency symbols if present\n    currency_symbols = [\'₹\', \'Rs\', \'INR\', \'₹\', \'₹ \', \'Rs \', \'INR \']\n    for symbol in currency_symbols:\n        s = s.replace(symbol, \'\').strip()\n\n    # Remove commas\n    s = s.replace(\',\', \'\')\n\n    # Convert to float\n    try:\n        return float(s)\n    except ValueError:\n        raise ValueError(f"Cannot parse price from \'{s}\'")\n\n# Example usage:\nprice_string = "₹ 12,345.67"\nparsed_price = parse_price(pr

In [64]:
def reflexion_loop(task: str, max_attempts: int = 10):
    """Reflexion: attempt -> feedback -> reflect into MEMORY -> next attempt uses lessons."""
    memory: list[str] = []                       # the Memory Node from the deck

    for i in range(1, max_attempts + 1):
        lessons = ("\nLESSONS FROM PREVIOUS FAILED ATTEMPTS:\n" +
                   "\n".join(f"- {m}" for m in memory)) if memory else ""
        code_out = llm(task + lessons, system="You are a Python developer.", temperature=0.8)
        print(f"code-output \n{code_out}")
        ok, fb = run_tests(code_out)             # external verifiable feedback
        print(f"[reflexion {i}] passed={ok}" + (f" | lessons in memory: {len(memory)}"))
        if ok: return code_out, i, memory

        # --- REFLECT: compress the failure into a short lesson (the key mechanic) ---
        lesson = llm(
            f"""Your code attempt failed these tests:
{fb}

The code was:
{code_out}

Write ONE compact lesson (max 30 words) stating WHY it failed and WHAT strategy
the next attempt should use. Be specific and technical, not generic advice.""",
            system="You extract root-cause lessons from failures.", temperature=0.2)
        memory.append(lesson.strip())

    return code_out, max_attempts, memory

In [65]:
solution, attempts, lessons = reflexion_loop(CODING_TASK)

code-output 
```python
import re

def parse_price(s: str) -> float:
    s = s.strip().replace('₹', '').replace('Rs.', '').replace(' ', '')
    s = re.sub(r'(?<=\d),(?=\d{3})', '', s)  # Remove commas in Indian format
    return float(s)
```

import re

def parse_price(s: str) -> float:
    s = s.strip().replace('₹', '').replace('Rs.', '').replace(' ', '')
    s = re.sub(r'(?<=\d),(?=\d{3})', '', s)  # Remove commas in Indian format
    return float(s)

[reflexion 1] passed=False | lessons in memory: 0
code-output 
```python
import re

def parse_price(s: str) -> float:
    # Remove currency symbols and strip whitespace
    s = s.replace('₹', '').replace('Rs.', '').strip()
    # Remove commas using the proper regex for Indian format
    s = re.sub(r'(?<=\d)(?=(\d{2})+(?!\d))', '', s.replace(',', ''))
    # Convert to float and return
    return float(s)
```

import re

def parse_price(s: str) -> float:
    # Remove currency symbols and strip whitespace
    s = s.replace('₹', '').replace(

In [67]:
solution = """
import re

def parse_price(s: str) -> float:
    # Remove currency symbols and strip whitespace
    s = s.replace('₹', '').replace('Rs.', '').strip()
    # Remove commas using the proper regex for Indian format
    s = re.sub(r'(?<=\d)(?=(\d{2})+(?!\d))', '', s.replace(',', ''))
    # Convert to float and return
    return float(s)
    """

exec(solution)
print(parse_price("9,99.0"))

999.0


In [49]:
print(f"\nSolved in {attempts} attempt(s). Lessons accumulated:")


Solved in 1 attempt(s). Lessons accumulated:


In [50]:
for l in lessons: print("  🧠", l)

In [51]:
exec(solution)

SyntaxError: invalid syntax (<string>, line 1)